# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking
**Document:** The Digital Personal Data Protection Act, 2023 (India) - a legal statute full of exceptions, prohibitions and role-specific rules.

**Pipeline 1 (Baseline):** Bi-Encoder -> FAISS cosine -> Top 10 -> Top 3 -> LLM -> Answer 1
**Pipeline 2 (Re-Rank):** same Top 10 -> Cross-Encoder -> Top 3 -> LLM (identical prompt) -> Answer 2

Both pipelines read the **same chunked corpus** and use the **same prompt, model and decoding**; the retrieved context is the only variable.

**Kaggle setup (just Run All):**
1. Settings -> **Internet ON**
2. Settings -> Accelerator -> **GPU T4 x2 (or P100)**
3. Add-ons -> Secrets -> add **`GROQ_API_KEY`** and tick *Attach to this notebook* (free key from https://console.groq.com). Without it the notebook falls back to a local GPU model automatically.

## 1. Setup & Installs

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate bitsandbytes openai google-genai pandas requests tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.4 MB/s eta 0:00:00


In [2]:
import os, re, time, glob
import numpy as np, pandas as pd, faiss, requests
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch
print("GPU available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 220)

GPU available: True


## 2. Configuration & document download
The PDF is fetched automatically from MeitY (official source). If the download is blocked, upload the Act as a Kaggle Dataset (any `.pdf` under `/kaggle/input` is picked up automatically).

In [3]:
PDF_URLS = [
    "https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf",
    "https://www.meity.gov.in/writereaddata/files/Digital%20Personal%20Data%20Protection%20Act%202023.pdf",
]
PDF_PATH = "/kaggle/working/dpdp_act_2023.pdf"
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36"}

BI_ENCODER    = "BAAI/bge-small-en-v1.5"                 # dense bi-encoder
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # cross-encoder re-ranker
MAX_WORDS, OVERLAP_WORDS = 200, 40
TOP_K_RETRIEVE, TOP_K_FINAL = 10, 3

if not os.path.exists(PDF_PATH):
    for u in PDF_URLS:
        try:
            r = requests.get(u, headers=HEADERS, timeout=90)
            if r.ok and r.content[:4] == b"%PDF":
                open(PDF_PATH, "wb").write(r.content); print("Downloaded from:", u); break
            print("Not a PDF / bad status from", u, r.status_code)
        except Exception as e:
            print("Download failed:", u, str(e)[:100])
if not os.path.exists(PDF_PATH):
    found = glob.glob("/kaggle/input/**/*.pdf", recursive=True)
    if found: PDF_PATH = found[0]; print("Using uploaded PDF:", PDF_PATH)
assert os.path.exists(PDF_PATH), "Could not download the PDF. Upload the DPDP Act PDF as a Kaggle Dataset and re-run."

Downloaded from: https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf


## 3. Extract, clean and parse into sections and sub-sections
The DPDP Act is organised into 44 numbered sections (each with sub-sections `(1)`, `(2)`...) plus a penalty Schedule. Instead of cutting blind fixed-width windows (which splits a rule from its exception and shreds the penalty table), we parse the Act **section by section**, then split each section at its sub-section boundaries, and keep the Schedule whole. This deliberately places a general rule and its "except when..." clause in **separate, self-labelled chunks** (e.g. Section 9(1) consent vs 9(4)/9(5) exemptions) - which is exactly the retrieval failure mode the assignment asks us to expose.

In [4]:
# --- 3a. Extract raw text, dropping page furniture and non-ASCII (bilingual) lines ---
reader = PdfReader(PDF_PATH)
raw_pages = [(p.extract_text() or "") for p in reader.pages]

def clean_line(ln):
    ln = ln.strip()
    if not ln: return ""
    ascii_ratio = sum(c.isascii() for c in ln) / max(len(ln), 1)
    if ascii_ratio < 0.85:                       # drop Hindi / garbled gazette lines
        return ""
    if re.search(r"THE GAZETTE OF INDIA", ln, re.I): return ""
    if re.match(r"^SEC\.?\s*1\]", ln): return ""
    if re.match(r"^\d+\s+THE GAZETTE", ln): return ""
    return ln

full = "\n".join(clean_line(l) for pg in raw_pages for l in pg.splitlines())
full = re.sub(r"[ \t]+", " ", full)
print("Clean text:", len(full), "chars")

# --- 3b. Split into the 44 sections using a sequence guard (N must follow N-1) ---
# A section starts as "N. ..." at the start of a line; the sequence guard rejects
# stray numbers (list items, cross-references like "section 7") that are not real headings.
token = re.compile(r"(?m)^\s*(\d{1,2})\.\s")
marks = []
expected = 1
for m in token.finditer(full):
    n = int(m.group(1))
    if n == expected:
        marks.append((n, m.start()))
        expected += 1

sched_m = re.search(r"THE SCHEDULE", full)
end_of_body = sched_m.start() if sched_m else len(full)

sections = []   # {num, text}
for i, (n, start) in enumerate(marks):
    end = marks[i + 1][1] if i + 1 < len(marks) else end_of_body
    body = full[start:end].strip()
    if body:
        sections.append({"num": n, "text": body})
print(f"Sections parsed: {len(sections)} (expected up to 44)")

Clean text: 62667 chars
Sections parsed: 44 (expected up to 44)


In [5]:
# --- 3c. Parse the penalty Schedule into clean "breach -> penalty" rows ---
# In the PDF the Schedule is a table that pypdf flattens into a column of breach
# descriptions followed by a column of penalty amounts. We re-pair them so the
# LLM sees e.g. "...security safeguards ... May extend to two hundred and fifty crore rupees"
# instead of a jumble. If the shapes don't line up we fall back to the raw block.
schedule_chunks = []
start_m = re.search(r"Breach in observing the obligation of Data Fiduciary", full)
if start_m and sched_m and start_m.start() < sched_m.start():
    # The penalty table sits just BEFORE the "THE SCHEDULE" heading; flatten line wraps first.
    flat = re.sub(r"\s+", " ", full[start_m.start():sched_m.start()])
    breaches  = re.findall(r"Breach [^.]*?(?:section \d+|thereunder)\.", flat)
    penalties = re.findall(r"May extend to .*?rupees\.|Up to the extent.*?instituted\.", flat)
    if breaches and len(breaches) == len(penalties):
        for b, p in zip(breaches, penalties):
            schedule_chunks.append(f"[Schedule - penalty] {b.strip()} Penalty: {p.strip()}")
    else:
        schedule_chunks.append("[Schedule - penalties] " + flat.strip())
    print(f"Schedule rows parsed: {len(schedule_chunks)} (breaches={len(breaches)}, penalties={len(penalties)})")
elif sched_m:
    schedule_chunks.append("[Schedule - penalties] " + re.sub(r"\s+", " ", full[sched_m.start():]).strip())
    print("Schedule heading found but table not matched - kept raw block")
else:
    print("Schedule not found - continuing without it")

Schedule rows parsed: 7 (breaches=7, penalties=7)


In [6]:
# --- 3d. Split each section at its sub-section boundaries, then window into <=MAX_WORDS chunks ---
def squash(s):                                   # for robust gold-chunk matching
    return re.sub(r"[^a-z0-9]", "", s.lower())

def split_subsections(num, body):
    """Split a section body at '(1)','(2)',... markers using a sequence guard, so cross-references
    like '(5) of section 8' do not trigger a false split. Sections without numbered sub-sections
    (e.g. the definitions in s.2) stay whole."""
    positions, expected = [], 1
    for m in re.finditer(r"\((\d{1,2})\)\s", body):
        if int(m.group(1)) == expected:
            positions.append((expected, m.start())); expected += 1
    if not positions:
        return [(f"Section {num}", body)]
    head = body[:positions[0][1]].strip()        # text before "(1)" belongs to the section head
    out = []
    for j, (k, pos) in enumerate(positions):
        end = positions[j + 1][1] if j + 1 < len(positions) else len(body)
        seg = body[pos:end].strip()
        if j == 0 and head:
            seg = head + " " + seg
        out.append((f"Section {num}({k})", seg))
    return out

chunks = []
def add_chunk(text, section, label):
    chunks.append({"id": len(chunks), "section": section, "label": label, "text": text, "sq": squash(text)})

for s in sections:
    for label, seg in split_subsections(s["num"], s["text"]):
        words = seg.split(); start = 0
        while start < len(words):
            piece = " ".join(words[start:start + MAX_WORDS])
            add_chunk(f"[{label}] " + piece, s["num"], label)
            if start + MAX_WORDS >= len(words):
                break
            start += MAX_WORDS - OVERLAP_WORDS
for sc in schedule_chunks:
    add_chunk(sc, "Schedule", "Schedule")

print(f"Total chunks: {len(chunks)} | words/chunk avg: {np.mean([len(c['text'].split()) for c in chunks]):.0f}")
print(chunks[0]["text"][:200])
assert len(chunks) > 40, "Too few chunks - section parse likely failed; inspect `full` above."

Total chunks: 149 | words/chunk avg: 77
[Section 1(1)] 1. (1) This Act may be called the Digital Personal Data Protection Act, 2023.


## 4. Self-test (verify preprocessing before building the index)
Every downstream claim rests on the corpus being parsed correctly. These checks confirm the sections were found, the Schedule penalties survived, and the headline failure mode - a general rule separated from its exception - is actually reachable.

In [7]:
def contains(probe):
    sq = squash(probe)
    return [c["id"] for c in chunks if sq in c["sq"]]

tests = []
def check(name, ok, detail=""):
    tests.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" :: {detail}" if detail else ""))

print("=== A. structure")
nums = [s["num"] for s in sections]
check("sections start at 1 and are contiguous", nums == list(range(1, len(nums) + 1)), f"1..{nums[-1] if nums else 0}")
check("at least 40 of 44 sections parsed", len(sections) >= 40, f"n={len(sections)}")
check("every chunk carries a [Section/Schedule] label", all(c["text"].startswith("[") for c in chunks))

print("\n=== B. key operative text landed somewhere")
for probe in ["voluntarily provided her personal data",          # s.7 legitimate uses
              "behavioural monitoring of children",               # s.9 prohibition
              "two hundred and fifty crore rupees",               # Schedule penalty
              "ten thousand rupees",                              # Schedule penalty
              "verifiable consent of the parent"]:                # s.9 children
    check(f"present: {probe!r}", bool(contains(probe)), f"chunks={contains(probe)[:3]}")

print("\n=== C. critical: rule and exception are in retrievable units")
prohib = contains("Data Fiduciary shall not undertake tracking")   # s.9(3) prohibition
excep  = contains("exempt from the applicability")                 # s.9(5)/17 exemption wording
check("a prohibition chunk exists", bool(prohib), f"{prohib[:3]}")
check("an exemption chunk exists",  bool(excep),  f"{excep[:3]}")

print("\n" + "=" * 60)
assert all(tests), "Self-test failed - fix parsing before trusting the results below."
print("SELF-TEST: all checks passed")

=== A. structure
  [PASS] sections start at 1 and are contiguous :: 1..44
  [PASS] at least 40 of 44 sections parsed :: n=44
  [PASS] every chunk carries a [Section/Schedule] label

=== B. key operative text landed somewhere
  [PASS] present: 'voluntarily provided her personal data' :: chunks=[23]
  [PASS] present: 'behavioural monitoring of children' :: chunks=[41]
  [PASS] present: 'two hundred and fifty crore rupees' :: chunks=[140, 141, 142]
  [PASS] present: 'ten thousand rupees' :: chunks=[141, 146]
  [PASS] present: 'verifiable consent of the parent' :: chunks=[39]

=== C. critical: rule and exception are in retrievable units
  [PASS] a prohibition chunk exists :: [41]
  [PASS] an exemption chunk exists :: [43]

SELF-TEST: all checks passed


## 5. Build vector index (Bi-Encoder + FAISS inner product on normalised vectors = cosine)

In [8]:
bi = SentenceTransformer(BI_ENCODER, device=DEVICE)
t0 = time.perf_counter()
emb = bi.encode([c["text"] for c in chunks], batch_size=64, normalize_embeddings=True,
                show_progress_bar=True, convert_to_numpy=True).astype("float32")
index_build_s = time.perf_counter() - t0
print(f"Indexing time: {index_build_s:.1f}s")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
assert index.ntotal == len(chunks)
assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-4)   # unit norm => inner product == cosine

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # recommended for bge retrieval
ce = CrossEncoder(CROSS_ENCODER, device=DEVICE, max_length=512)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 6. Retrieval functions (with timing)

In [9]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    t0 = time.perf_counter()
    q = bi.encode([QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    dt = time.perf_counter() - t0
    return [int(i) for i in ids[0]], [float(s) for s in scores[0]], dt

def rerank(query, cand_ids):
    t0 = time.perf_counter()
    pairs = [(query, chunks[i]["text"]) for i in cand_ids]
    ce_scores = ce.predict(pairs, show_progress_bar=False)
    dt = time.perf_counter() - t0
    order = np.argsort(-ce_scores)
    return [cand_ids[j] for j in order], [float(ce_scores[j]) for j in order], dt

## 7. LLM generation (identical prompt for both pipelines)
`LLM_BACKEND = "groq"` uses the free Groq API (key as Kaggle secret `GROQ_API_KEY`); the notebook auto-detects which model your key can use. If no key is found it falls back to a local open-weights model on the GPU (`"local"`). `"gemini"` is also supported.

In [10]:
LLM_BACKEND = "groq"                        # "groq" (default, falls back to local if no key) | "local" | "gemini"
LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct"    # used only on fallback; set 7B + LOAD_4BIT for better answers
LOAD_4BIT   = False
GEMINI_MODEL = "gemini-2.5-flash"

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

if LLM_BACKEND == "groq" and not get_secret("GROQ_API_KEY"):
    print("!! GROQ_API_KEY not found (Add-ons -> Secrets, tick 'Attach to this notebook'). Falling back to LOCAL model.")
    LLM_BACKEND = "local"
print("LLM backend in use:", LLM_BACKEND)

if LLM_BACKEND == "local":
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    kw = dict(device_map="auto")
    if LOAD_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    llm = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, **kw)
elif LLM_BACKEND == "groq":
    from openai import OpenAI
    oai = OpenAI(api_key=get_secret("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")
    available = sorted(m.id for m in oai.models.list().data)
    print("Models available to your key:", available)
    preferred = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.1-8b-instant"]
    GROQ_MODEL = next((m for m in preferred if m in available), None)
    assert GROQ_MODEL, f"No preferred model enabled for your key. Available: {available}"
    print("Using Groq model:", GROQ_MODEL)
elif LLM_BACKEND == "gemini":
    from google import genai
    from google.genai import types
    gem = genai.Client(api_key=get_secret("GEMINI_API_KEY"))

PROMPT = """You are a careful assistant answering questions about a legal document.
Use ONLY the context below. If the context does not contain the answer, say "Not found in the provided context."
Pay close attention to exceptions, restrictions, prohibitions and conditions, and quote the Section number you relied on.

Context:
{context}

Question: {question}

Answer concisely:"""

def build_context(ids):
    return "\n\n".join(f"[Chunk {i} | section {chunks[i]['section']}]\n{chunks[i]['text']}" for i in ids)

def generate(query, ids, retries=4):
    prompt = PROMPT.format(context=build_context(ids), question=query)
    for a in range(retries):
        try:
            if LLM_BACKEND == "local":
                text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                               tokenize=False, add_generation_prompt=True)
                inputs = tok(text, return_tensors="pt").to(llm.device)
                with torch.no_grad():
                    out = llm.generate(**inputs, max_new_tokens=300, do_sample=False)
                return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            if LLM_BACKEND == "groq":
                r = oai.chat.completions.create(model=GROQ_MODEL, temperature=0,
                        messages=[{"role": "user", "content": prompt}])
                return r.choices[0].message.content.strip()
            r = gem.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0))
            return r.text.strip()
        except Exception as e:
            msg = str(e); print("LLM error:", msg[:160])
            if any(c in msg for c in ("404", "401", "403", "does not exist")):
                raise
            time.sleep(5 * (a + 1))
    return "LLM_ERROR"

print("LLM smoke test:", generate("What is 2+2?", []))

LLM backend in use: groq
Models available to your key: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']
Using Groq model: openai/gpt-oss-120b
LLM smoke test: Not found in the provided context.


## 8. Run both pipelines for one query + rank-shift analysis

In [11]:
def run_query(query, gold, ans_kw, category="", verbose=True):
    """gold = phrases (any case/spacing) that must ALL appear in the true answer chunk; used to track ranks.
       ans_kw = keywords; if ANY appears in the LLM answer we mark it as containing the key fact (heuristic)."""
    cand, bi_scores, t_bi = baseline_retrieve(query)
    ranked, ce_scores, t_ce = rerank(query, cand)
    base_top3, rr_top3 = cand[:TOP_K_FINAL], ranked[:TOP_K_FINAL]
    gold_sq = [squash(g) for g in gold]

    rows = []
    for r_b, cid in enumerate(cand, 1):
        r_r = ranked.index(cid) + 1
        rows.append({"chunk": cid, "section": chunks[cid]["section"], "baseline_rank": r_b,
                     "bi_score": round(bi_scores[r_b-1], 3), "ce_score": round(ce_scores[r_r-1], 3),
                     "rerank_rank": r_r, "shift": r_b - r_r,
                     "gold": all(g in chunks[cid]["sq"] for g in gold_sq)})
    df = pd.DataFrame(rows).sort_values("rerank_rank")
    g = df[df.gold]
    b_rank = int(g.baseline_rank.min()) if len(g) else None
    r_rank = int(g.rerank_rank.min()) if len(g) else None
    if b_rank is None:               status = "gold chunk not in top-10"
    elif b_rank > 3 and r_rank <= 3: status = "PROMOTED into top-3"
    elif b_rank <= 3 and r_rank <= 3: status = "in top-3 for both"
    elif b_rank <= 3 and r_rank > 3:  status = "DEMOTED out of top-3"
    else:                            status = "missed by both"

    ans1 = generate(query, base_top3); ans2 = generate(query, rr_top3)
    def has_kw(a): return any(k in a.lower() for k in ans_kw)
    out = {"category": category, "query": query, "df": df, "t_bi": t_bi, "t_ce": t_ce,
           "base_top3": base_top3, "rr_top3": rr_top3, "b_rank": b_rank, "r_rank": r_rank, "status": status,
           "gold_chunk": int(g.sort_values("rerank_rank").chunk.iloc[0]) if len(g) else None,
           "gold_section": g.sort_values("rerank_rank").section.iloc[0] if len(g) else None,
           "ans1": ans1, "ans2": ans2, "ans1_ok": has_kw(ans1), "ans2_ok": has_kw(ans2)}
    if verbose:
        print("=" * 100); print(f"[{category}] {query}")
        print(df.to_string(index=False))
        print(f">> STATUS: {status}" + (f" | gold chunk {out['gold_chunk']} (section {out['gold_section']}): Rank #{b_rank} -> Rank #{r_rank}" if len(g) else ""))
        print(f"\n--- Answer 1 (Baseline, chunks {base_top3}) ---\n{ans1}")
        print(f"\n--- Answer 2 (Re-ranked, chunks {rr_top3}) ---\n{ans2}")
    return out

## 9. Failure-mode stress tests (pre-built query bank for the DPDP Act)
Each query targets a failure type the assignment names: **keyword trap / synonym distractor**, **exceptions & corner cases**, **negation & constraints**, and **buried clauses** (answer at baseline rank 4-10). Two queries are **recall-ceiling** cases whose answer the bi-encoder never puts in the top 10 - kept deliberately to show that *re-ranking repairs ordering, not recall*. Two are **controls** the baseline should already get right.

In [12]:
# (category, query, gold phrases that must appear in the answer chunk, keywords expected in a correct answer)
QUERY_BANK = [
 ("Keyword trap", "Which kind of company has to hire an independent auditor and a privacy officer living in India?",
      ["independent data auditor"], ["auditor", "significant"]),
 ("Keyword trap", "Do new small businesses get any relief from the obligations in this law?",
      ["startup"], ["startup"]),
 ("Exception / corner case", "When can personal data be used without taking the person's consent?",
      ["voluntarily provided her personal data"], ["voluntar", "legitimate"]),
 ("Exception / corner case", "Which obligations are relaxed when data is processed only for research, archiving or statistics?",
      ["research, archiving or statistical"], ["research", "archiv", "statistic"]),
 ("Exception / corner case", "Does the law still apply to a foreign customer's data handled by an Indian outsourcing firm?",
      ["pursuant to any contract entered into with any person outside the territory"], ["not apply", "exempt", "does not"]),
 ("Negation / constraint", "What is a company prohibited from doing with children's data?",
      ["behavioural monitoring"], ["track", "behavioural", "advertis"]),
 ("Negation / constraint", "What must a person NOT do while giving her data or lodging a complaint?",
      ["frivolous"], ["frivolous", "false", "impersonat"]),
 ("Negation / constraint", "Can a child's data be handled without the approval of a parent or guardian?",
      ["verifiable consent of the parent"], ["verifiable", "parent", "guardian"]),
 ("Buried clause", "How large can the fine be if a company fails to protect data with proper security safeguards?",
      ["two hundred and fifty crore"], ["250", "two hundred and fifty"]),
 ("Buried clause", "What is the maximum penalty if an individual breaches her own duties under the Act?",
      ["ten thousand rupees"], ["10,000", "10000", "ten thousand"]),
 ("Buried clause", "Can personal data be sent to other countries?",
      ["restrict the transfer"], ["restrict", "notif"]),
 ("Buried clause", "In which languages must the privacy notice be made available?",
      ["Eighth Schedule"], ["eighth schedule", "english"]),
 ("Buried clause", "Can a person appoint someone to exercise her rights if she dies or becomes incapacitated?",
      ["nominate"], ["nominat"]),
 # ---- synonym-gap probes: layperson wording that avoids the statute's vocabulary (rescue candidates) ----
 ("Synonym gap", "If an app keeps using my information after I ask it to stop, when must it erase my data?",
      ["erase personal data, upon the Data Principal withdrawing her consent"], ["erase", "withdraw"]),
 ("Synonym gap", "If a company hires another firm to handle my data, who stays responsible for following the law?",
      ["irrespective of any agreement to the contrary"], ["data fiduciary", "responsible"]),
 # ---- mined rescues (gold at baseline rank 4-10, promoted into top-3 by the Cross-Encoder; see 9.0c) ----
 ("Buried clause", "If a company mishandles my data, where must I complain before going to the Board?",
      ["exhaust the opportunity of redressing her grievance"], ["grievance", "before", "board"]),
 ("Keyword trap", "What investigative powers does the Data Protection Board have, like a court?",
      ["same powers as are vested in a civil court"], ["civil court", "summon", "inquiry"]),
 ("Negation / constraint", "Can I take a data dispute to an ordinary civil court?",
      ["No civil court shall have the jurisdiction"], ["no civil court", "jurisdiction", "bar"]),
 ("Control", "What are the grounds on which a person may process personal data?",
      ["lawful purpose"], ["consent", "lawful"]),
 ("Control", "Who is a Data Principal under this Act?",
      ["the individual to whom the personal data relates"], ["individual"]),
]

results = []
for cat, q, gold, kws in QUERY_BANK:
    results.append(run_query(q, gold, kws, category=cat))

[Keyword trap] Which kind of company has to hire an independent auditor and a privacy officer living in India?
 chunk  section  baseline_rank  bi_score  ce_score  rerank_rank  shift  gold
    45       10              1     0.620    -2.492            1      0  True
    58       16              8     0.549    -7.926            2      6 False
    57       16             10     0.548    -9.334            3      7 False
    62       17              6     0.552    -9.964            4      2 False
    63       17              3     0.570   -10.008            5     -2 False
    25        7              9     0.549   -10.805            6      3 False
    36        8              2     0.583   -10.966            7     -5 False
    27        7              5     0.557   -11.055            8     -3 False
     3        2              4     0.566   -11.156            9     -5 False
    51       13              7     0.552   -11.292           10     -3 False
>> STATUS: in top-3 for both | gold chunk 

### 9.0 Gold-chunk uniqueness guard
Each query's `gold` phrases must resolve to a chunk from a **single section**, so a loose phrase cannot manufacture a rank-shift "win" on the wrong unit. This assertion pins every gold label before the results are trusted.

In [13]:
multi = []
for cat, q, gold, kws in QUERY_BANK:
    ids = [c["id"] for c in chunks if all(squash(g) in c["sq"] for g in gold)]
    secs = sorted({str(chunks[i]["section"]) for i in ids})
    assert ids, f"gold probes for {q!r} matched NO chunk - fix the probe"
    flag = "ok " if len(secs) == 1 else "MULTI"
    if len(secs) != 1: multi.append((q, secs))
    print(f"  {flag} section {','.join(secs):>10}  <- {q[:58]}")
if multi:
    print(f"\nNote: {len(multi)} probe(s) resolve to several sections (operative text legitimately appears in more "
          "than one place, e.g. scope + exemption). Rank tracking uses the first gold chunk; not an error.")
else:
    print("\nGold uniqueness guard: every probe pins to a single section.")

  ok  section         10  <- Which kind of company has to hire an independent auditor a
  ok  section         17  <- Do new small businesses get any relief from the obligation
  ok  section          7  <- When can personal data be used without taking the person's
  ok  section         17  <- Which obligations are relaxed when data is processed only 
  ok  section         17  <- Does the law still apply to a foreign customer's data hand
  ok  section          9  <- What is a company prohibited from doing with children's da
  MULTI section      15,28  <- What must a person NOT do while giving her data or lodging
  ok  section          9  <- Can a child's data be handled without the approval of a pa
  MULTI section 44,Schedule  <- How large can the fine be if a company fails to protect da
  MULTI section 44,Schedule  <- What is the maximum penalty if an individual breaches her 
  ok  section         16  <- Can personal data be sent to other countries?
  MULTI section        5,6  <- In whi

### 9.0b Query provenance - where does each gold chunk start out?
Re-ranking can only help when the gold chunk sits at baseline rank **4-10** (inside the candidate window but outside the top 3). We bucket every query by its baseline band so the set is justified by evidence, not hand-picked: band `1-3` has no headroom, `4-10` is where a rescue is possible, `>10` is a recall ceiling the cross-encoder can never fix.

In [14]:
def band_of(rank):
    if rank is None: return ">10 (recall ceiling)"
    if rank <= 3:    return "1-3 (no headroom)"
    return "4-10 (rescue possible)"

prov = pd.DataFrame([{"query": r["query"][:60], "category": r["category"],
                      "baseline_rank": r["b_rank"], "band": band_of(r["b_rank"])} for r in results])
print(prov.to_string(index=False))
print("\nBand distribution:"); print(prov["band"].value_counts().to_string())

                                                       query                category  baseline_rank                   band
Which kind of company has to hire an independent auditor and            Keyword trap            1.0      1-3 (no headroom)
Do new small businesses get any relief from the obligations             Keyword trap            1.0      1-3 (no headroom)
When can personal data be used without taking the person's c Exception / corner case            6.0 4-10 (rescue possible)
Which obligations are relaxed when data is processed only fo Exception / corner case            3.0      1-3 (no headroom)
Does the law still apply to a foreign customer's data handle Exception / corner case            3.0      1-3 (no headroom)
What is a company prohibited from doing with children's data   Negation / constraint            1.0      1-3 (no headroom)
What must a person NOT do while giving her data or lodging a   Negation / constraint            4.0 4-10 (rescue possible)
Can a child's da

### 9.0c Query mining - screen HARD candidates for rescues (no LLM, fast)
The bank above is mostly easy (gold already at rank 1-3), so the Cross-Encoder has little room to help. To find genuine rescues we screen a pool of **harder** questions - procedural, lay-vocabulary queries whose answer text uses different words than the question - and keep only those where the gold chunk sits at baseline rank 4-10 **and** the Cross-Encoder promotes it into the top 3. This is retrieval-only, so it runs in seconds with no API calls. Move any `*** RESCUE ***` winner into `QUERY_BANK` and re-run Section 9 to generate its answers.

In [15]:
HARD_POOL = [
 ("Can the government let its own agencies skip this law for national security reasons?",
      ["friendly relations with foreign States"]),
 ("If a company keeps breaking the rules, can its website be blocked for the public?",
      ["block for access by the public"]),
 ("If a company mishandles my data, where must I complain before going to the Board?",
      ["exhaust the opportunity of redressing her grievance"]),
 ("What investigative powers does the Data Protection Board have, like a court?",
      ["same powers as are vested in a civil court"]),
 ("Can I take a data dispute to an ordinary civil court?",
      ["No civil court shall have the jurisdiction"]),
 ("Are officials protected if they act honestly while enforcing the Act?",
      ["anything which is done or intended to be done in good faith"]),
 ("Can a late appeal still be accepted if I had a good reason for the delay?",
      ["sufficient cause for not preferring the appeal"]),
 ("Where does the money from penalties go?",
      ["credited to the Consolidated Fund of India"]),
]

def screen(query, gold):
    cand, bi_s, _ = baseline_retrieve(query)
    ranked, _, _ = rerank(query, cand)
    gold_sq = [squash(g) for g in gold]
    def rank_in(order):
        for i, cid in enumerate(order, 1):
            if all(g in chunks[cid]["sq"] for g in gold_sq): return i, cid
        return None, None
    b, cid = rank_in(cand); r, _ = rank_in(ranked)
    if b is None:       status = "gold not in top-10 (recall ceiling)"
    elif b > 3 and r is not None and r <= 3: status = "*** RESCUE ***"
    elif b <= 3:        status = "no headroom (already top-3)"
    elif r is not None and r > 3: status = "in window, not rescued"
    else:               status = "?"
    return b, r, (chunks[cid]["section"] if cid is not None else None), status

print(f"{'baseline':>8} {'rerank':>7} {'section':>8}  status / query")
print("-" * 100)
winners = 0
for q, gold in HARD_POOL:
    b, r, sec, st = screen(q, gold)
    if st.startswith("***"): winners += 1
    print(f"{str(b):>8} {str(r):>7} {str(sec):>8}  {st:<32} {q[:48]}")
print(f"\nRescue candidates found: {winners}. Move each '*** RESCUE ***' query (with its gold probe and an "
      "answer-keyword list) into QUERY_BANK, then re-run Section 9 to generate Answer 1 vs Answer 2.")

baseline  rerank  section  status / query
----------------------------------------------------------------------------------------------------
       3       4       17  no headroom (already top-3)      Can the government let its own agencies skip thi
       1       1       37  no headroom (already top-3)      If a company keeps breaking the rules, can its w
       8       1       13  *** RESCUE ***                   If a company mishandles my data, where must I co
       5       2       28  *** RESCUE ***                   What investigative powers does the Data Protecti
       7       1       39  *** RESCUE ***                   Can I take a data dispute to an ordinary civil c
       1       1       35  no headroom (already top-3)      Are officials protected if they act honestly whi
       1       2       29  no headroom (already top-3)      Can a late appeal still be accepted if I had a g
       1       1       34  no headroom (already top-3)      Where does the money from penaltie

### 9.1 Summary: rank shifts and answer correctness for every query

In [16]:
summary = pd.DataFrame([{
    "category": r["category"], "query": r["query"][:70], "gold_chunk": r["gold_chunk"], "section": r["gold_section"],
    "baseline_rank": r["b_rank"], "rerank_rank": r["r_rank"], "status": r["status"],
    "Answer1_has_key_fact": r["ans1_ok"], "Answer2_has_key_fact": r["ans2_ok"]} for r in results])
summary

,category,query,gold_chunk,section,baseline_rank,rerank_rank,status,Answer1_has_key_fact,Answer2_has_key_fact
0,Keyword trap,Which kind of company has to hire an independent auditor and a privacy,45.0,10,1.0,1.0,in top-3 for both,True,True
1,Keyword trap,Do new small businesses get any relief from the obligations in this la,63.0,17,1.0,10.0,DEMOTED out of top-3,True,False
2,Exception / corner case,When can personal data be used without taking the person's consent?,23.0,7,6.0,5.0,missed by both,True,True
3,Exception / corner case,Which obligations are relaxed when data is processed only for research,62.0,17,3.0,1.0,in top-3 for both,True,True
4,Exception / corner case,Does the law still apply to a foreign customer's data handled by an In,59.0,17,3.0,4.0,DEMOTED out of top-3,True,True
5,Negation / constraint,What is a company prohibited from doing with children's data?,41.0,9,1.0,1.0,in top-3 for both,True,True
6,Negation / constraint,What must a person NOT do while giving her data or lodging a complaint,56.0,15,4.0,1.0,PROMOTED into top-3,False,True
7,Negation / constraint,Can a child's data be handled without the approval of a parent or guar,39.0,9,1.0,1.0,in top-3 for both,True,True
8,Buried clause,How large can the fine be if a company fails to protect data with prop,142.0,Schedule,1.0,1.0,in top-3 for both,True,True
9,Buried clause,What is the maximum penalty if an individual breaches her own duties u,146.0,Schedule,2.0,1.0,in top-3 for both,False,True


In [17]:
print("Rank shifts where re-ranking changed the gold position:\n")
for r in results:
    if r["b_rank"] is not None and r["b_rank"] != r["r_rank"]:
        print(f"[{r['category']}] chunk {r['gold_chunk']} (section {r['gold_section']}): Rank #{r['b_rank']} -> Rank #{r['r_rank']}  | {r['status']}")
        print(f"    Q: {r['query']}")

promoted = [r for r in results if r["status"].startswith("PROMOTED")]
print(f"\nQueries where the Cross-Encoder PROMOTED the answer chunk into the top-3: {len(promoted)}")

# Report over ALL queries (a gold chunk never retrieved counts as a miss, rank -> infinity for MRR),
# so the hardest recall-ceiling cases are NOT dropped from the denominator.
N = len(summary)
base_hit3 = (summary.baseline_rank <= 3).sum()      # NaN <= 3 is False -> ceiling counts as a miss
rr_hit3   = (summary.rerank_rank  <= 3).sum()
def mrr_all(col):
    return float(np.mean([1.0 / r if pd.notna(r) else 0.0 for r in summary[col]]))
print(f"\nAggregate over ALL {N} queries (recall-ceiling cases counted as misses, not dropped):")
print(f"  Hit@3  baseline: {base_hit3}/{N} = {base_hit3/N:.2f} | re-ranked: {rr_hit3}/{N} = {rr_hit3/N:.2f}")
print(f"  MRR    baseline: {mrr_all('baseline_rank'):.3f} | re-ranked: {mrr_all('rerank_rank'):.3f}")
print(f"  Answers containing key fact  baseline: {summary.Answer1_has_key_fact.mean():.2f} | re-ranked: {summary.Answer2_has_key_fact.mean():.2f}")

Rank shifts where re-ranking changed the gold position:

[Keyword trap] chunk 63 (section 17): Rank #1 -> Rank #10  | DEMOTED out of top-3
    Q: Do new small businesses get any relief from the obligations in this law?
[Exception / corner case] chunk 23 (section 7): Rank #6 -> Rank #5  | missed by both
    Q: When can personal data be used without taking the person's consent?
[Exception / corner case] chunk 62 (section 17): Rank #3 -> Rank #1  | in top-3 for both
    Q: Which obligations are relaxed when data is processed only for research, archiving or statistics?
[Exception / corner case] chunk 59 (section 17): Rank #3 -> Rank #4  | DEMOTED out of top-3
    Q: Does the law still apply to a foreign customer's data handled by an Indian outsourcing firm?
[Negation / constraint] chunk 56 (section 15): Rank #4 -> Rank #1  | PROMOTED into top-3
    Q: What must a person NOT do while giving her data or lodging a complaint?
[Buried clause] chunk 146 (section Schedule): Rank #2 -> Rank #1  | 

### 9.2 Answer 1 vs Answer 2 for the rescued (promoted) queries - copy these into your report

In [18]:
for r in promoted:
    print("=" * 100); print(f"[{r['category']}] {r['query']}")
    print(f"Gold chunk {r['gold_chunk']} (section {r['gold_section']}): Rank #{r['b_rank']} -> Rank #{r['r_rank']}")
    print(f"\nAnswer 1 (baseline top-3 = {r['base_top3']}):\n{r['ans1']}")
    print(f"\nAnswer 2 (re-ranked top-3 = {r['rr_top3']}):\n{r['ans2']}\n")

[Negation / constraint] What must a person NOT do while giving her data or lodging a complaint?
Gold chunk 56 (section 15): Rank #4 -> Rank #1

Answer 1 (baseline top-3 = [24, 8, 38]):
Not found in the provided context.

Answer 2 (re-ranked top-3 = [56, 38, 8]):
According to **Section 15**, a Data Principal must **not** :

- impersonate another person when providing her personal data for a specified purpose;  
- suppress any material information when providing her personal data for any document, unique identifier, proof of identity or proof of address issued by the State or its instrumentalities; and  
- register a false or frivolous grievance or complaint with a Data Fiduciary or the Board.

[Buried clause] If a company mishandles my data, where must I complain before going to the Board?
Gold chunk 53 (section 13): Rank #8 -> Rank #1

Answer 1 (baseline top-3 = [97, 33, 10]):
Not found in the provided context.

Answer 2 (re-ranked top-3 = [53, 11, 97]):
You must first pursue redress u

> The `Answer_has_key_fact` columns are a quick keyword heuristic only. For the report, read Answer 1 vs Answer 2 yourself and state whether the baseline hallucinated, omitted a clause, or misstated a rule.

## 10. Latency profiling

In [19]:
def profile(queries, n_repeat=5):
    bi_t, ce_t = [], []
    c0, _, _ = baseline_retrieve(queries[0]); rerank(queries[0], c0)      # warm-up
    for q in queries:
        for _ in range(n_repeat):
            cand, _, tb = baseline_retrieve(q); _, _, tc = rerank(q, cand)
            bi_t.append(tb); ce_t.append(tc)
    return np.array(bi_t), np.array(ce_t)

qs = [q for _, q, _, _ in QUERY_BANK]
bi_t, ce_t = profile(qs)
lat = pd.DataFrame({
    "Stage": ["Pipeline 1: baseline retrieval (embed query + FAISS top-10)",
              "Cross-Encoder re-rank of 10 (query, chunk) pairs",
              "Pipeline 2 total retrieval (baseline + re-rank)"],
    "Mean (ms)":   [1000*bi_t.mean(), 1000*ce_t.mean(), 1000*(bi_t+ce_t).mean()],
    "Median (ms)": [1000*np.median(bi_t), 1000*np.median(ce_t), 1000*np.median(bi_t+ce_t)],
    "Std (ms)":    [1000*bi_t.std(), 1000*ce_t.std(), 1000*(bi_t+ce_t).std()]}).round(2)
print(f"Device: {DEVICE} | {len(qs)} queries x 5 repeats | corpus = {len(chunks)} chunks")
print(f"Re-ranking overhead: x{(bi_t+ce_t).mean()/bi_t.mean():.1f} the baseline retrieval time")
lat

Device: cuda | 20 queries x 5 repeats | corpus = 149 chunks
Re-ranking overhead: x3.2 the baseline retrieval time


,Stage,Mean (ms),Median (ms),Std (ms)
0,Pipeline 1: baseline retrieval (embed query + FAISS top-10),12.36,12.18,1.43
1,"Cross-Encoder re-rank of 10 (query, chunk) pairs",27.58,27.97,3.51
2,Pipeline 2 total retrieval (baseline + re-rank),39.94,40.08,4.00


### Scaling experiment: Cross-Encoder over the ENTIRE corpus vs. only 10 candidates

In [20]:
q = qs[0]
ce.predict([(q, chunks[0]["text"])] * 8, show_progress_bar=False)           # warm-up
t0 = time.perf_counter(); ce.predict([(q, c["text"]) for c in chunks], batch_size=64, show_progress_bar=False)
t_full = time.perf_counter() - t0
per_pair = t_full / len(chunks)
print(f"Cross-Encoder on ALL {len(chunks)} chunks: {t_full:.2f}s | on 10 candidates: {1000*ce_t.mean():.1f} ms")
print(f"Bi-encoder indexed the whole corpus once in {index_build_s:.1f}s and answers each query in ~{1000*bi_t.mean():.0f} ms.")
for N in [10_000, 100_000, 1_000_000]:
    print(f"Extrapolated CE over {N:,} chunks ({DEVICE}): ~{per_pair*N/60:.1f} min PER QUERY")

Cross-Encoder on ALL 149 chunks: 0.33s | on 10 candidates: 27.6 ms
Bi-encoder indexed the whole corpus once in 1.3s and answers each query in ~12 ms.
Extrapolated CE over 10,000 chunks (cuda): ~0.4 min PER QUERY
Extrapolated CE over 100,000 chunks (cuda): ~3.7 min PER QUERY
Extrapolated CE over 1,000,000 chunks (cuda): ~36.8 min PER QUERY


## 11. Auto-generated tables for the report

In [21]:
print("### Rank-shift table (paste into report 11.2)\n")
rt = summary[["category", "query", "gold_chunk", "section", "baseline_rank", "rerank_rank", "status"]]
print(rt.to_markdown(index=False))
print("\n### Latency table (paste into report 11.3)\n")
print(lat.to_markdown(index=False))
summary.to_csv("rank_shift_summary.csv", index=False); lat.to_csv("latency.csv", index=False)

### Rank-shift table (paste into report 11.2)

| category                | query                                                                  |   gold_chunk | section   |   baseline_rank |   rerank_rank | status                   |
|:------------------------|:-----------------------------------------------------------------------|-------------:|:----------|----------------:|--------------:|:-------------------------|
| Keyword trap            | Which kind of company has to hire an independent auditor and a privacy |           45 | 10        |               1 |             1 | in top-3 for both        |
| Keyword trap            | Do new small businesses get any relief from the obligations in this la |           63 | 17        |               1 |            10 | DEMOTED out of top-3     |
| Exception / corner case | When can personal data be used without taking the person's consent?    |           23 | 7         |               6 |             5 | missed by both           |
| Except

## 12. Report

### 12.1 Architecture comparison
**Bi-Encoder:** query and passage are encoded *independently* by the same transformer into fixed-size vectors; relevance is cosine similarity / dot product. Passage vectors are pre-computed offline and indexed, so search is a fast nearest-neighbour lookup. Because query and passage never attend to each other, one vector must compress the whole passage's meaning, so fine details (negation, "except when...", exact constraints, role limits) get blurred.

**Cross-Encoder:** query and passage are concatenated (`[CLS] query [SEP] passage [SEP]`) and run through the transformer *together*, so self-attention operates jointly across all query and passage tokens. It outputs a single relevance score directly. It captures token-level interactions (negation, qualifiers, synonyms) far more accurately, but nothing can be pre-computed: every (query, passage) pair needs its own forward pass.

### 12.2 Qualitative & rank-shift analysis
Use the auto-generated table in Section 11 and the Answer 1 vs Answer 2 printout in Section 9.2. For each rescued query write: (a) "Chunk X (Section N) moved from Rank #a -> Rank #b"; (b) whether Answer 1 hallucinated, omitted an essential clause, or misstated a rule because the critical chunk sat outside the top-3; (c) how Answer 2 fixed it. Cover one example each of keyword trap, exception/corner case, negation/constraint and buried clause, and discuss the **recall-ceiling** queries (gold never in the top 10) as evidence that the cross-encoder cannot recover a chunk stage 1 never retrieved, and the **regression/demotion** case if one occurs.

### 12.3 Latency profiling
Paste the latency table from Section 11 and name the device and corpus size. The Cross-Encoder adds a small constant overhead for 10 pairs, which is typically far smaller than the LLM generation time.

### 12.4 Scaling question: why not run the Cross-Encoder over the whole collection?
A Cross-Encoder cannot pre-compute anything: each (query, chunk) pair needs a full transformer forward pass at query time, so cost grows as **O(N)** forward passes per query (N = number of chunks), and nothing can be cached because the score depends on the query. For millions of chunks this means minutes to hours per query (see Section 10). A Bi-Encoder does its expensive work once offline, and at query time needs one forward pass for the query plus an (approximate) nearest-neighbour search, sublinear with ANN indices such as HNSW/IVF. Hence the standard design: **cheap, high-recall retrieval** (Bi-Encoder, top-k) followed by **expensive, high-precision re-ranking** (Cross-Encoder) on only k candidates.